# PCA on MNIST

KNN on 784 pixel columns, then on a few principal components; pictures of the digits in 2D and 3D; choosing the number of components.

In [1]:
import time
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.datasets import fetch_openml
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

## The data

MNIST: 70,000 handwritten digits of 28 x 28 pixels, one column per pixel. The first download takes a minute; after that it is read from a local cache. We keep a random 42,000 images, the size of Kaggle's Digit Recognizer `train.csv`.

In [2]:
X, y = fetch_openml("mnist_784", version=1, return_X_y=True, as_frame=False)
idx = np.random.default_rng(0).choice(len(X), 42000, replace=False)
X, y = X[idx], y[idx].astype(int)
X.shape   # (42000, 784)

(42000, 784)

In [3]:
# one row back into a 28 x 28 picture
px.imshow(X[0].reshape(28, 28), color_continuous_scale="gray_r", title=f"label: {y[0]}")

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)
X_train.shape, X_test.shape

((33600, 784), (8400, 784))

## KNN on all 784 columns

In [5]:
start = time.perf_counter()
knn = KNeighborsClassifier(n_neighbors=5).fit(X_train, y_train)
acc = accuracy_score(y_test, knn.predict(X_test))
print(f"784 columns: accuracy {acc:.3f}, {time.perf_counter() - start:.1f} s")

784 columns: accuracy 0.968, 120.3 s


## KNN on 50 principal components

PCA runs on the raw pixels: it centres them itself, and all pixels share one unit (brightness 0 to 255), so no scaling is needed.

In [6]:
start = time.perf_counter()
pca = PCA(n_components=50).fit(X_train)          # learn the components from the training set only
Z_train, Z_test = pca.transform(X_train), pca.transform(X_test)
knn = KNeighborsClassifier(n_neighbors=5).fit(Z_train, y_train)
acc = accuracy_score(y_test, knn.predict(Z_test))
print(f"50 components: accuracy {acc:.3f}, {time.perf_counter() - start:.1f} s")

50 components: accuracy 0.973, 288.1 s


IOStream.flush timed out


IOStream.flush timed out


## Accuracy as k grows (a few values; each takes a few seconds)

In [7]:
rows = []
for k in [1, 2, 3, 5, 10, 20, 50, 100, 300]:
    p = PCA(n_components=k).fit(X_train)
    pred = KNeighborsClassifier(n_neighbors=5).fit(p.transform(X_train), y_train).predict(p.transform(X_test))
    rows.append({"k": k, "accuracy": accuracy_score(y_test, pred)})
pd.DataFrame(rows)

,k,accuracy
0,1,0.271190
1,2,0.425595
2,3,0.489048
3,5,0.731071
4,10,0.925476
5,20,0.966548
6,50,0.972500
7,100,0.971310
8,300,0.967738


## Pictures: 2D and 3D (drag the 3D plot to turn it; click a digit in the legend to hide it)

In [8]:
p3 = PCA(n_components=3).fit(X_train)
Z = p3.transform(X_test)
df = pd.DataFrame({"PC1": Z[:, 0], "PC2": Z[:, 1], "PC3": Z[:, 2], "digit": y_test.astype(str)})
order = {"digit": [str(i) for i in range(10)]}
px.scatter(df, x="PC1", y="PC2", color="digit", category_orders=order, opacity=0.5)

In [9]:
fig = px.scatter_3d(df, x="PC1", y="PC2", z="PC3", color="digit", category_orders=order, opacity=0.6)
fig.update_traces(marker_size=2)
fig

## What PCA learned

In [10]:
print(p3.explained_variance_)        # the 3 largest eigenvalues
print(p3.explained_variance_ratio_)  # their share of the total variance
print(p3.components_.shape)          # (3, 784): 3 eigenvectors, each with 784 numbers

[331120.72427897 245936.58484081 211044.85650333]
[0.09654705 0.07170935 0.06153574]
(3, 784)


In [11]:
# each component is itself a 28 x 28 picture: which pixels it combines
px.imshow(p3.components_[0].reshape(28, 28), color_continuous_scale="RdBu", title="PC1 as an image")

## Choosing k: keep 90% of the variance

In [12]:
full = PCA().fit(X_train)
cum = np.cumsum(full.explained_variance_ratio_)
k90 = int(np.searchsorted(cum, 0.90) + 1)
print("components for 90% of the variance:", k90)
px.line(x=np.arange(1, 785), y=cum, labels={"x": "k", "y": "cumulative explained variance"})

components for 90% of the variance: 87


In [13]:
# scikit-learn can pick k for us: a number between 0 and 1 means "keep this share of the variance"
PCA(n_components=0.90).fit(X_train).n_components_

np.int64(87)

## Checks used in the Note: ink and PC1, standardising the pixels, later components

In [14]:
# PC1 against the number of non-zero (inked) pixels in each test image
ink = (X_test > 0).sum(axis=1)
print('correlation of PC1 with ink:', np.corrcoef(Z[:, 0], ink)[0, 1].round(2))

knn5 = lambda a, b: KNeighborsClassifier(n_neighbors=5).fit(a, y_train).score(b, y_test)
# standardising every pixel first (each pixel gets mean 0 and standard deviation 1)
scaler = StandardScaler().fit(X_train)
X_train_s, X_test_s = scaler.transform(X_train), scaler.transform(X_test)
print('standardised, 784 columns:', knn5(X_train_s, X_test_s))
q = PCA(n_components=50).fit(X_train_s)
print('standardised, PCA 50:', knn5(q.transform(X_train_s), q.transform(X_test_s)))
# drop pixels that are non-zero in fewer than 5% of training images, then standardise the rest
keep = (X_train > 0).mean(axis=0) >= 0.05
print(keep.sum(), 'pixels kept, standardised, accuracy', knn5(X_train_s[:, keep], X_test_s[:, keep]))

# Are the later components just noise for KNN? Compare them with a copy whose values are shuffled
# across images: same spread, but no link to the label any more.
A, B = full.transform(X_train), full.transform(X_test)
rng = np.random.default_rng(1)
def shuffled(M, lo, hi):
    M = M[:, :hi].copy()
    for j in range(lo, hi):
        M[:, j] = rng.permutation(M[:, j])
    return M
print('components 51-784 alone:', knn5(A[:, 50:], B[:, 50:]))
print('first 50 + real 51-784:', knn5(A, B))
print('first 50 + shuffled 51-784:', knn5(shuffled(A, 50, 784), shuffled(B, 50, 784)))
print('components 101-300 alone:', knn5(A[:, 100:300], B[:, 100:300]))
print('first 100 + shuffled 101-300:', knn5(shuffled(A, 100, 300), shuffled(B, 100, 300)))


correlation of PC1 with ink: 0.86


standardised, 784 columns: 0.9404761904761905


standardised, PCA 50: 0.9544047619047619


397 pixels kept, standardised, accuracy 0.9591666666666666


components 51-784 alone: 0.5427380952380952


first 50 + real 51-784: 0.9676190476190476


first 50 + shuffled 51-784: 0.9698809523809524


components 101-300 alone: 0.41


first 100 + shuffled 101-300: 0.9705952380952381
